In [1]:
import numpy as np
import struct
import matplotlib.pyplot as plt
with open('../input/emnist_source_files/emnist-bymerge-train-images-idx3-ubyte','rb') as f:
    magic, size = struct.unpack(">II", f.read(8))
    nrows, ncols = struct.unpack(">II", f.read(8))
    train_data = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
    train_data = train_data.reshape((size,nrows,ncols))
print('Train images shape : ',np.shape(train_data))
with open('../input/emnist_source_files/emnist-bymerge-test-images-idx3-ubyte','rb') as f:
    magic, size = struct.unpack(">II", f.read(8))
    nrows, ncols = struct.unpack(">II", f.read(8))
    test_data = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
    test_data = test_data.reshape((size,nrows,ncols))
print('Test images shape : ',np.shape(test_data))
with open('../input/emnist_source_files/emnist-bymerge-train-labels-idx1-ubyte','rb') as f:
    magic, size = struct.unpack(">II", f.read(8))
    train_labels = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
    train_labels = train_labels.reshape((size,))
print('Train labels shape : ',np.shape(train_labels))
with open('../input/emnist_source_files/emnist-bymerge-test-labels-idx1-ubyte','rb') as f:
    magic, size = struct.unpack(">II", f.read(8))
    test_labels = np.fromfile(f, dtype=np.dtype(np.uint8).newbyteorder('>'))
    test_labels = test_labels.reshape((size,))
print('Test labels shape : ',np.shape(test_labels))

class_map = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabdefghnqrt'
number_of_classes = len(class_map)
print('Class size : ', number_of_classes)

Train images shape :  (697932, 28, 28)
Test images shape :  (116323, 28, 28)
Train labels shape :  (697932,)
Test labels shape :  (116323,)
Class size :  47


In [ ]:
def show_img(data, labels, row_num):
    img_flip = np.transpose(data[row_num], axes=[1,0]) # img_size * img_size arrays
    plt.title('Class: ' + str(labels[row_num]) + ', Label: ' + str(class_map[labels[row_num]]))
    plt.imshow(img_flip, cmap='Greys_r')
print("Test correct image correspondence with label")
show_img(test_data, test_labels, 604)

In [2]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Data normalisation
train_data = train_data / 255.0
test_data = test_data / 255.0

train_data_size = train_data.shape[0]
train_data_height = 28
train_data_width = 28
train_data_img_size = train_data_height*train_data_width

train_data = train_data.reshape(train_data_size, train_data_height, train_data_width, 1)

test_data_size = test_data.shape[0]
test_data_height = 28
test_data_width = 28
test_data_img_size = test_data_height*test_data_width

test_data = test_data.reshape(test_data_size, test_data_height, test_data_width, 1)

# Transform labels
train_labels = to_categorical(train_labels, number_of_classes)
test_labels = to_categorical(test_labels, number_of_classes)
# Split some data for validation
train_data, validation_data, train_labels, validation_labels = train_test_split(train_data, train_labels, test_size=0.2, random_state=42)

print("###### Final shapes ######")
print("Train data shape : ", np.shape(train_data), "\tTrain labels shape : ", np.shape(train_labels))
print("Test data shape : ", np.shape(test_data), "\t\tTest labels shape : ", np.shape(test_labels))
print("Validation data shape : ", np.shape(validation_data), "\tValidation labels shape : ", np.shape(validation_labels))

2024-02-10 22:51:18.381069: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-10 22:51:18.381113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-10 22:51:18.421170: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-10 22:51:18.492081: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-02-10 22:51:19.443933: W tensorflow/compiler/tf2

###### Final shapes ######
Train data shape :  (558345, 28, 28, 1) 	Train labels shape :  (558345, 47)
Test data shape :  (116323, 28, 28, 1) 		Test labels shape :  (116323, 47)
Validation data shape :  (139587, 28, 28, 1) 	Validation labels shape :  (139587, 47)


In [3]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,3,input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D(2,2),
    #################################################
    tf.keras.layers.Flatten(input_shape=(28,28,1)),
    tf.keras.layers.Dense(512,activation='relu'),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(number_of_classes, activation='softmax')
])

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=tf.keras.metrics.CategoricalAccuracy())

es = EarlyStopping(monitor='val_categorical_accuracy', mode='max', verbose=1, patience=175)
mc = ModelCheckpoint('combined_emnist_model.h5', monitor='val_categorical_accuracy', mode='max', verbose=1, save_best_only=True, patience=175)
history = model.fit(train_data, train_labels, epochs=100, validation_data= (validation_data,validation_labels) , callbacks=[es,mc], verbose=0)
with open('history.json', 'w') as f:
    json.dump(history.history, f)

2024-02-10 22:51:27.919587: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-10 22:51:28.173245: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-02-10 22:51:28.173454: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.